# CodePause Phase 5 — Dataset v5 Recovery + Streaming Demo

**Goal:** recover from Phase 4 collapse with high-entropy v5 data and strict code validation.

**Final verified outcome on Colab T4:** PASS.
- Base + `best_phase3_prompt`: `1/30`
- Adapter v5 + `best_phase3_prompt`: `3/30`
- Delta: `+2/30`

This proves a minimal controlled improvement. It does **not** prove production robustness yet.

## 1. Setup

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git'
PROJECT_DIR = pathlib.Path('/content/AMDh')

os.chdir('/content')
if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a git checkout')

os.chdir(PROJECT_DIR)
print('cwd:', pathlib.Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

## 2. Tests

In [ ]:
!python -m pytest tests/test_generate_v5.py tests/test_training_normalization.py -v

## 3. Dependencies

In [ ]:
import subprocess, sys
deps = ['trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'transformers', 'torchao>=0.16.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
print('deps installed ok')

## 4. Baseline — base + best prompt

In [ ]:
!python eval/evaluate_baseline.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --problems_path data/heldout_problems_30.jsonl \
  --output_path results/phase5_base_best_prompt.jsonl \
  --prompt_template best_phase3_prompt

## 5. Train adapter v5

In [ ]:
!python training/sft_lora.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --dataset_path data/thinkanywhere_sft_v5.jsonl \
  --output_dir results/sft_phase5_v5 \
  --epochs 1 \
  --max_steps 150 \
  --max_seq_length 1024 \
  --device auto \
  --load_in_4bit \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05

## Phase 6 Recovery — Rebuild lost adapter and export release candidate

This section recreates the Phase 5 adapter from the Phase 5 v5 dataset and recipe. The original Phase 5 runtime artifact was lost during a Colab reset. This is not Phase 8.

In [ ]:
# 1. Verify GPU
!nvidia-smi

# 2. Setup Repo (if needed, but usually we are already inside it)
import os, pathlib, subprocess, sys
print(f"Current working directory: {pathlib.Path.cwd()}")

# 3. Install required packages
deps = ['transformers', 'datasets', 'accelerate', 'peft', 'trl', 'bitsandbytes', 'safetensors']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
print("Required packages installed.")

# 4. Run training/sft_lora.py with Phase 5 recipe
!python training/sft_lora.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --dataset_path data/thinkanywhere_sft_v5.jsonl \
  --output_dir results/sft_phase5_v5 \
  --epochs 1 \
  --max_steps 150 \
  --max_seq_length 1024 \
  --device auto \
  --load_in_4bit \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05

# 5. Export Release Candidate
import shutil
rc_dir = pathlib.Path('results/phase6_release_candidate')
rc_dir.mkdir(parents=True, exist_ok=True)

# Copy adapter artifacts
adapter_src = pathlib.Path('results/sft_phase5_v5')
if (adapter_src / 'adapter_config.json').exists():
    shutil.copy(adapter_src / 'adapter_config.json', rc_dir / 'adapter_config.json')
if (adapter_src / 'adapter_model.safetensors').exists():
    shutil.copy(adapter_src / 'adapter_model.safetensors', rc_dir / 'adapter_model.safetensors')
elif (adapter_src / 'adapter_model.bin').exists():
    shutil.copy(adapter_src / 'adapter_model.bin', rc_dir / 'adapter_model.bin')

# 6. Write Metadata and Reports (Packaging only, no post-training inference)
import json

metadata = {
    "phase": 6,
    "base_model": "Qwen/Qwen1.5-1.8B-Chat",
    "recipe": "phase5_v5_recovery",
    "status": "rebuilt_from_loss"
}
with open(rc_dir / 'metadata_model.json', 'w') as f:
    json.dump(metadata, f, indent=2)

with open(rc_dir / 'README_MODEL.md', 'w') as f:
    f.write("# Phase 6 Rebuilt Model\n\nThis adapter was rebuilt from Phase 5 v5 recipe after artifact loss.")

report_note = "Packaging cell did not run post-training inference to save time; verification should use standard evaluation scripts."
with open(rc_dir / 'load_test_report.md', 'w') as f:
    f.write(f"# Load Test Report\n\n{report_note}")
with open(rc_dir / 'inference_smoke_report.md', 'w') as f:
    f.write(f"# Inference Smoke Report\n\n{report_note}")
with open(rc_dir / 'eval_report.md', 'w') as f:
    f.write(f"# Eval Report\n\n{report_note}")

# 7. Generate checksums
!cd results/phase6_release_candidate && find . -type f -not -name 'checksums.txt' -exec sha256sum {} + > checksums.txt

# 8. Create ZIPs
zip_name = "codepause_phase6_release_candidate_REBUILT"
# /content version
!zip -r /content/{zip_name}.zip results/phase6_release_candidate
# repo-local version
!zip -r {zip_name}.zip results/phase6_release_candidate

print(f"Recovery complete. Artifacts in {rc_dir} and {zip_name}.zip")